# 22DM015 Final Project — Financial PhraseBank
## Person C: Part 2 — Zero-Shot LLM track

**Dataset:** `takala/financial_phrasebank`, config `sentences_allagree` (2,264 sentences).‍
**Labels:** 0 = negative, 1 = neutral, 2 = positive.‍

### Shared data contract (set by Person D — do NOT re-split)
- Splits are committed under `data/` as CSVs: `train` (1584), `val` (227), `test` (453), `labeled_32` (32).‍
- The **32 labelled** examples are a balanced sample from train (11 neg / 10 neu / 11 pos).‍
- Part 2 'unlabelled' data = train minus the 32 (`unlabeled_pool`).‍
- Evaluate headline numbers on **`test`** only; tune on **`val`**.‍ Use `eval_utils.evaluate` so we're comparable.‍
- Log every result with `eval_utils.log_result(...)` into `results/results.csv`.‍

> **AI disclosure:** any AI-generated code/text in this notebook must be watermarked and declared (course rule).‍ Interpretation, methodology justification, and analysis must be student-authored.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# --- Reproducibility seed (required by the assignment) ---
import os, random, sys
import numpy as np
import pandas as pd
SEED = 618
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Make the shared helpers importable (they live in the repo root, one level up).
sys.path.append(os.path.abspath('..'))
import data_utils as du
import eval_utils as eu

splits = du.load_splits()            # identical train/val/test for everyone
train, val, test = splits['train'], splits['val'], splits['test']
# labeled_32 is no longer part of load_splits — load the committed CSV directly.
labeled_32 = pd.read_csv('../data/labeled_32.csv')
# Part 2 'unlabelled' data = train rows not among the 32 labelled ones.
pool = train[~train['id'].isin(set(labeled_32['id']))].reset_index(drop=True)
for k, v in {**splits, 'labeled_32': labeled_32}.items():
    print(f'{k:11s} {len(v):5d}', dict(v['label_name'].value_counts()))

## Part 2c.‍ Zero-Shot Learning with an LLM (0.5)
Classify the **test** set zero-shot (no training examples in the prompt).‍ Map free-text LLM output to {negative, neutral, positive} robustly.‍ Document the prompt.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 2c: zero-shot classification of the shared test split (453 sentences) with Claude.
# - Exact model id is recorded in LLM_MODEL and logged (course rule: declare LLM + version).
# - Every raw API reply is cached in ../.cache/llm_responses_<model>.csv (COMMITTED — it is
#   the exact-reproduction artifact for these rows: no API key or nondeterminism needed).
#   The cache key includes a fingerprint of the prompt + system prompt, so editing either
#   forces fresh API calls instead of silently serving replies from the old prompt.
# - RESUME-AWARE: if the 'zero-shot' row already exists in results.csv, nothing is called;
#   flushes are atomic and every 50 calls, so an interrupted run resumes for free.
# - PACED for tier-1 rate limits (50 requests/min, 30k input tokens/min on opus): one
#   sequential call every `min_interval` seconds; the SDK additionally retries 429s.
#   Expect ~12 min for the zero-shot pass, ~22 min for few-shot.
# Needs the ANTHROPIC_API_KEY environment variable (console.anthropic.com -> API Keys).
import os, hashlib, time

import pandas as pd

try:
    import anthropic
    HAVE_SDK = True
except ImportError:           # pip install anthropic
    HAVE_SDK = False

LLM_MODEL = 'claude-opus-4-8'        # exact model id, declared per course rules
CACHE_CSV = f'../.cache/llm_responses_{LLM_MODEL}.csv'
LABELS = ['negative', 'neutral', 'positive']   # index == label id (shared contract)

ZERO_SHOT_PROMPT = (
    'Classify the sentiment of this financial sentence as exactly one of: '
    'negative, neutral, positive. Reply with only that word.\n\nSentence: {text}'
)


def logged(method):
    """Latest TEST row for (LLM_MODEL, method) from the shared scoreboard. The eval module
    keys rows on (model, method, split, n_train_labeled) and no longer tracks person, so
    we match on LLM_MODEL + method here."""
    if not eu.RESULTS_CSV.exists():
        return None
    r = pd.read_csv(eu.RESULTS_CSV)
    r = r[(r['model'] == LLM_MODEL) & (r['method'] == method) & (r['split'] == 'test')]
    return {k: r.iloc[-1][k] for k in eu.METRIC_KEYS} if len(r) else None


def parse_label(reply):
    """Map free-text LLM output to a label id robustly; None if no label word found."""
    hits = [(reply.lower().find(w), i) for i, w in enumerate(LABELS)]
    hits = [(pos, i) for pos, i in hits if pos >= 0]
    return min(hits)[1] if hits else None     # first label word mentioned wins


def _sha1(s):
    return hashlib.sha1(s.encode('utf-8')).hexdigest()


def _flush_cache(new):
    """Merge new replies into the cache CSV atomically (tmp + replace)."""
    add = pd.DataFrame([{'kind': k[0], 'key': k[1], 'raw': v} for k, v in new.items()])
    prev = (pd.read_csv(CACHE_CSV, keep_default_na=False)
            if os.path.exists(CACHE_CSV) else pd.DataFrame())
    merged = (pd.concat([prev, add], ignore_index=True)
              .drop_duplicates(subset=['kind', 'key'], keep='last'))
    os.makedirs(os.path.dirname(CACHE_CSV), exist_ok=True)
    merged.to_csv(CACHE_CSV + '.tmp', index=False)
    os.replace(CACHE_CSV + '.tmp', CACHE_CSV)


def classify_split(kind, df, system=None, prompt=ZERO_SHOT_PROMPT, min_interval=1.4):
    """Classify df['text'] sequentially, paced to stay under the org rate limit.
    Replies are cached under (kind:prompt-fingerprint, sha1(text)); the cache is
    flushed every 50 calls and on failure, so an interrupted run resumes for free."""
    ns = f"{kind}:{_sha1(prompt + chr(0) + (system or ''))[:8]}"   # prompt-aware namespace
    client = anthropic.Anthropic(max_retries=10)   # 429s also back off per retry-after
    cache = {}
    if os.path.exists(CACHE_CSV):
        prev = pd.read_csv(CACHE_CSV, keep_default_na=False)
        cache = {(r['kind'], r['key']): str(r['raw']) for _, r in prev.iterrows()}

    new, raws, last = {}, [], 0.0
    try:
        for i, text in enumerate(df['text']):
            k = (ns, _sha1(text))
            if k in cache:
                raws.append(cache[k])
                continue
            wait = min_interval - (time.monotonic() - last)
            if wait > 0:
                time.sleep(wait)
            last = time.monotonic()
            resp = client.messages.create(
                model=LLM_MODEL, max_tokens=16, system=system or anthropic.NOT_GIVEN,
                messages=[{'role': 'user', 'content': prompt.format(text=text)}],
            )
            raw = next((b.text for b in resp.content if b.type == 'text'), '')
            if raw.strip():               # never cache an empty reply — re-ask next run
                new[k] = raw
            raws.append(raw)
            if len(new) % 50 == 0:
                _flush_cache(new)
                print(f'  {i + 1}/{len(df)} classified ({kind})')
    finally:
        if new:                       # keep progress even if a call ultimately failed
            _flush_cache(new)
            print(f'[cache] {len(new)} new replies saved ({ns})')

    preds = [parse_label(r) for r in raws]
    n_bad = sum(p is None for p in preds)
    if n_bad > 0.05 * len(df):        # neutral-defaulting >5% would inflate accuracy
        raise RuntimeError(f'{n_bad}/{len(df)} replies unparsable — fix the prompt/parsing '
                           'instead of silently defaulting them to the majority class')
    if n_bad:
        print(f'[warn] {n_bad} unparsable replies -> defaulted to neutral (majority class)')
    return [1 if p is None else p for p in preds], n_bad


zs_m = logged('zero-shot')
if zs_m is not None:
    print('[cached]')
elif not HAVE_SDK:
    print('[waiting] anthropic SDK not installed — pip install anthropic, then re-run.')
elif not os.environ.get('ANTHROPIC_API_KEY'):
    print('[waiting] ANTHROPIC_API_KEY not set — set the env var and re-run this cell.')
else:
    zs_preds, zs_bad = classify_split('zero-shot', test)
    zs_m = eu.evaluate(test['label'].values, zs_preds)
    eu.log_result(LLM_MODEL, 'zero-shot', 0, zs_m,
                  notes=f'zero-shot on test (n=453); unparsable={zs_bad}')
    print('[done]')

if zs_m is not None:
    print('2c zero-shot TEST:', eu.fmt(zs_m))

### 2c - analysis

The zero-shot setup is deliberately minimal: a single instruction prompt asks the model to classify each test sentence as exactly one of `negative`, `neutral`, or `positive`, with no labelled examples and no fine-tuning.‍ Free-text replies are mapped back to the project's label encoding by a robust parser that takes the first label word mentioned, so trailing commentary like "Neutral.‍ (slightly positive bias)" is handled cleanly; the run logs zero unparsable replies on the 453-sentence test pass.‍

We found this numbers are the highest in the project.‍ `claude-opus-4-8` reaches 0.978 macro-F1 with zero labelled examples, above 32-shot BERT (0.601), augmented BERT (0.635), and even the LLM-generated 2d run (0.841).‍ The per-class numbers are uniformly strong: negative F1 = 0.968, neutral F1 = 0.987, positive F1 = 0.978.‍

There is no minority-class collapse, the same failure mode every BERT run below ~50 % data exhibits is simply absent.‍ The few-shot variant (32 in-context examples in the system prompt) logs byte-identical metrics to zero-shot: 0.978 macro-F1, identical per-class scores, identical accuracy.‍ Within the cache's reach, the 32 examples changed no test prediction.‍

With this methodology, we need to take in account this two things:

1.‍ benchmark contamination: Financial PhraseBank is a public dataset (Malo et al., 2014) and a frontier LLM has very likely seen it, or near-identical text, in pretraining; a high score could measure partly the recall of a memorised benchmark rather than pure generalisation.‍ The exact same caveat applies to the few-shot tie: the model is at ceiling because the test sentences are not novel to it.‍ A clean re-test would require an out-of-distribution split that did not exist at the model's training cut-off.‍

2.‍ cost and latency: each prediction is a paid API call subject to rate limits and ~1.4 s pacing per request, which costs about 12 minutes for the 453-sentence test pass and recurring spend for any production deployment, against a fine-tuned local BERT that runs offline at near-zero marginal cost.‍ Also, the BERT row at full data (0.959 macro-F1, Part 3) is auditable and deterministic, the LLM row is a black-box API.‍

The bottom line is the course's central zero-shot hypothesis realised on this dataset: a frontier LLM with no task-specific training beats every BERT result here, including the one trained on all 1,584 labels.‍ The data-efficiency reading is tested: BERT never catches the LLM, even at 100 % of the data.‍ Whether that gap is a fair measurement of LLM capability or an artefact of contamination is the right question to carry into Part 3.‍

## (Optional) Few-shot with the 32 labelled examples
Same LLM but put a few of `labeled_32` in the prompt as in-context examples; compare to zero-shot.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Optional few-shot: same LLM and parsing, but the 32 shared labelled examples
# (balanced 11/10/11) go in the system prompt as in-context examples.
# Comparable to 2a/2b/2d because n_train_labeled=32 — same information budget.
# Paced slower than 2c: the ~1.3k-token example prompt rides on every call, so the
# 30k input-tokens/min tier-1 limit binds (~21 calls/min -> min_interval=2.9s).
# The cache namespace fingerprints FEW_SHOT_SYSTEM, so a regenerated labeled_32
# invalidates these cached replies automatically (it changes the system prompt).
FEW_SHOT_SYSTEM = (
    'You classify the sentiment of financial sentences as negative, neutral, or positive.\n'
    'Here are labelled examples:\n\n'
    + '\n\n'.join(f'Sentence: {r.text}\nLabel: {r.label_name}' for r in labeled_32.itertuples())
    + '\n\nReply with only the label word.'
)

fs_m = logged('few-shot')
if fs_m is not None:
    print('[cached]')
elif not HAVE_SDK or not os.environ.get('ANTHROPIC_API_KEY'):
    print('[waiting] needs the anthropic SDK + ANTHROPIC_API_KEY (see 2c above).')
else:
    fs_preds, fs_bad = classify_split('few-shot', test, system=FEW_SHOT_SYSTEM, min_interval=2.9)
    fs_m = eu.evaluate(test['label'].values, fs_preds)
    eu.log_result(LLM_MODEL, 'few-shot', 32, fs_m,
                  notes=f'32 in-context examples; unparsable={fs_bad}')
    print('[done]')

if fs_m is not None:
    print('few-shot TEST:', eu.fmt(fs_m))
    if zs_m is not None:
        print('delta macro-F1 (few-shot - zero-shot): {:+.4f}'.format(
            float(fs_m['f1_macro']) - float(zs_m['f1_macro'])))

### 2 Few-shot - analysis

The optional few-shot variant uses the same LLM (`claude-opus-4-8`), the same parser, and the same test split as zero-shot, but injects the entire shared `labeled_32` set (11 negative + 10 neutral + 11 positive) into the system prompt as in-context examples, formatted `Sentence: <text>\nLabel: <label_name>`.‍ This is the standard in-context-learning setup discused in class: no gradient updates, no fine-tuning, only a prompt that shows the model what the task looks like before asking the question.‍

We can notice the result is a byte-identical tie with zero-shot: Both rows log the same accuracy (0.9823), the same macro-F1 (0.9779), and the same per-class F1s (negative 0.9683, neutral 0.9873, positive 0.9780).‍ Within the cache's reach the 32 in-context examples changed no test prediction at all: every one of the 453 test sentences was assigned the same label by the model whether or not the demonstrations were present.‍ That is itself the result, and it carries three concrete consequences worth spelling out separately.‍

1.‍ Ceiling effect: A tie at 0.978 is not "few-shot is worthless"; it means "zero-shot already saturates this benchmark for this model." Adding examples can only help when the model has uncertainty to resolve, and on `sentences_allagree` (the unanimously-labelled subset) `claude-opus-4-8` has none.‍ A weaker model (Haiku, an open 7B) would almost certainly not tie zero-shot to few-shot, because it would have wrong predictions for the demonstrations to correct.‍ The tie is therefore informative about this specific model on this specific data, not about in-context learning in general.‍

2.‍ Contamination signature, not proof: A perfect zero-shot ceiling is consistent with the contamination hypothesis (Financial PhraseBank is a public dataset from 2014, well within any frontier model's pretraining window).‍ The byte-identical tie is therefore a necessary but not sufficient sign of contamination; it is what we would expect if the model has memorised, but the same pattern can also appear in a genuinely easy task with no contamination at all.‍

3.‍ At-equal-budget contrast with 32-shot BERT (the central few-shot question): Zero-shot and few-shot LLM are tied at 0.978; 32-shot BERT (Part 2a) reaches only 0.601, augmented BERT (Part 2b) 0.635.‍ Same label budget (32 examples) almost identical model size budget if you compare opus to a 110 M-parameter BERT, but a +0.38 macro-F1 gap.‍ The label budget is not the bottleneck; the problem is what the model already knows about the task.‍ BERT has to fit a classifier head on those 32 examples, which is hopeless at this sample size; the LLM does not need to fit anything because it already encodes the task and the demonstrations only re-state what it already does.‍

Even after discounting the 0.978 for contamination, the gradient between zero-shot and few-shot still answers the question the course poses about in-context learning.‍ So, what closes the gap to BERT is new information (Part 2d's LLM-generated training data, +0.24 over 2a) or more training data (the Part 3 curve crossing 0.73 between 25 % and 50 %).‍ The decisive Part-2 lever is therefore data, not prompting.‍